In [63]:
import matplotlib.pyplot as plt
import nelpy as nel
import nelpy.plotting as npl
import numpy as np
import seaborn as sns
from neuro_py.io import loading
from neuro_py.plotting.figure_helpers import set_size
from neuro_py.tuning import maps
from skimage import measure
import os
import pandas as pd
import scipy.io as sio
import logging

logging.getLogger().setLevel(logging.ERROR)
basepath =  r"W:\data\PPP\PPP8\day12"
basename=os.path.basename(basepath)# Define the new folder path
#create working folder
new_folder_path = os.path.join(basepath, "complex_system")
brain_region="CA1"
animal="Rat"   #or Mouse
cell_type='all'
path_parts = basepath.split("\\")  # Split the path using backslash
animal_name = path_parts[-2]  # Second-to-last part is the animal name
day = path_parts[-1]  # Last part is the day
save_dir = r"D:\Documents\GitHub\R_Project\PPP\codebook\Criticality_analysis\complex_system\manifold"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
# Create the folder if it doesn't exist
if not os.path.exists(new_folder_path):
    os.makedirs(new_folder_path)
    print(f"Folder created: {new_folder_path}")
else:
    print(f"Folder already exists: {new_folder_path}")
# load in spike data from hpc pyramidal cells
st, cm = loading.load_spikes(
    basepath, brainRegion=brain_region                          # putativeCellType="Pyr"
)
# load theta epochs
state_dict = loading.load_SleepState_states(basepath)
nrem_epochs = nel.EpochArray(
    state_dict["NREMstate"],
)
awake_epochs = nel.EpochArray(
    state_dict["WAKEstate"],
)
theta_epochs = nel.EpochArray(
    state_dict["THETA"],
)
rem_epochs=nel.EpochArray(state_dict["REMstate"],)
try:
    # Try default loader
    epoch_df = loading.load_epoch(basepath)
except Exception as e:
    print(f"load_epoch failed with error: {e}")
    print("Falling back to manual .mat file load...")

    # Load the .mat file
    mat = sio.loadmat(os.path.join(basepath, "DayNight25.MergePoints.events.mat"), squeeze_me=True)

    # Extract the 'MergePoints' structure
    merge_points = mat['MergePoints']

    # Extract the timestamps array: shape (N, 2)
    timestamps = merge_points['timestamps']  # this is the field you're interested in
    # Unwrap the array inside
    timestamps = np.array(timestamps.tolist())

    # Now you should have shape (N, 2)
    epoch_df = pd.DataFrame(timestamps, columns=["startTime", "stopTime"])

sleep_epochs = nel.EpochArray(np.array([epoch_df.startTime, epoch_df.stopTime]).T)
E_spikes = st.time[(cm.putativeCellType == 'Pyramidal Cell')]
I_spikes= st.time[(cm.putativeCellType.str.contains("Interneuron"))]
spike_times=st.time
n_cell=len(spike_times)
print(f"number of cells: {n_cell}")

Folder already exists: W:\data\PPP\PPP8\day12\complex_system
number of cells: 92


In [64]:
# theta_epochs=theta_epochs[theta_epochs.durations>10]

In [65]:
import os, re, numpy as np, pandas as pd
import glob  # <-- add this
def _load_mat_any(mat_path):
    from scipy.io import loadmat
    try:
        return loadmat(
            mat_path,
            squeeze_me=True,
            struct_as_record=False,
            simplify_cells=True
        )
    except TypeError:
        return loadmat(
            mat_path,
            squeeze_me=True,
            struct_as_record=False
        )
    except NotImplementedError:
        import mat73
        return mat73.loadmat(mat_path)
def _to_1d_array(x):
    """
    Robustly coerce MATLAB-loaded field to a 1D numpy array.
    - Scalars -> 1-length array
    - Lists/tuples/ndarrays -> flattened 1D
    - None / missing -> empty array
    """
    if x is None:
        return np.array([], dtype=float)
    arr = np.asarray(x, dtype=object)  # start as object to avoid surprises
    if arr.ndim == 0:
        # scalar or unsized object
        return np.atleast_1d(arr).astype(float, copy=False)
    # flatten and try float cast (ignore non-numeric gracefully)
    try:
        return np.asarray(arr).ravel().astype(float, copy=False)
    except (ValueError, TypeError):
        # fallback: best-effort, convert elementwise
        return np.array([float(v) for v in np.ravel(arr)], dtype=float)

def _to_state_str(s):
    # MATLAB char arrays sometimes come in as numpy arrays
    try:
        if isinstance(s, (bytes, bytearray)):
            return s.decode('utf-8', 'ignore')
        if isinstance(s, str):
            return s
        # arrays or weird containers -> join characters
        arr = np.asarray(s)
        if arr.ndim == 0:
            return str(arr.item())
        # e.g., array of characters
        return ''.join(arr.tolist()) if arr.dtype.kind in ('U','S') else str(arr)
    except Exception:
        return str(s)

def load_d2_timeline_complete(basepath, region):
    """Complete version that loads ALL sleep-state data, including d2_err."""
    region = region.upper()
    search_dir = os.path.join(basepath, "complex_system")
    # pats = [f"*_{region}_D2_time_resolved_rippleablate.mat", f"*_{region}_D2_time_rippleablate.mat"]
    pats = [f"*_{region}_D2_time_resolved_error_bar.mat", f"*_{region}_D2_time_error_bar.mat"]
    files = []
    for pat in pats:
        files.extend(glob.glob(os.path.join(search_dir, pat)))
    if not files:
        raise FileNotFoundError(f"No *_{region}_D2_time*.mat under {search_dir}")

    mat_path = max(files, key=os.path.getmtime)
    print(f"Loading: {os.path.basename(mat_path)}")

    mat = _load_mat_any(mat_path)
    results = mat['results']
    params = mat.get('params', {})

    # Make results iterable no matter if it's a list, ndarray, or dict
    if isinstance(results, dict):
        iter_results = list(results.values())
    else:
        iter_results = list(results if isinstance(results, (list, tuple)) else np.atleast_1d(results))

    print(f"📊 Found {len(iter_results)} result items (sleep states)")
    print(f"📊 Parameters: {params}")

    # Try to infer animal/day from filename if available
    fname = os.path.basename(mat_path)
    animal_name = None
    day = None
    m_day = re.search(r'day(\d+)', fname, flags=re.I)
    if m_day:
        day = int(m_day.group(1))
    m_animal = re.match(r'([^_]+)_day\d+_', fname)
    if m_animal:
        animal_name = m_animal.group(1)

    all_centers, all_d2, all_d2err, all_states = [], [], [], []
    n_states_with_err = 0

    for i, result_item in enumerate(iter_results):
        # Access as dict-like regardless of MATLAB struct quirks
        state = _to_state_str(result_item.get('state', 'UNKNOWN')) if hasattr(result_item, 'get') else \
                _to_state_str(result_item['state']) if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) else 'UNKNOWN'

        centers_raw = result_item.get('center', []) if hasattr(result_item, 'get') else \
                      (result_item['center'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'center' in result_item else [])
        d2_raw = result_item.get('d2', []) if hasattr(result_item, 'get') else \
                 (result_item['d2'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'd2' in result_item else [])
        d2err_raw = result_item.get('d2_err', []) if hasattr(result_item, 'get') else \
                    (result_item['d2_err'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'd2_err' in result_item else [])

        centers  = _to_1d_array(centers_raw)
        d2_vals  = _to_1d_array(d2_raw)
        d2_err   = _to_1d_array(d2err_raw)

        # Presence flag (some states may not have d2_err)
        has_err = d2_err.size > 0 and np.any(np.isfinite(d2_err))
        n_states_with_err += int(has_err)

        # Harmonize lengths: truncate all to the common min
        sizes = [centers.size, d2_vals.size] + ([d2_err.size] if has_err else [])
        n_pts = min(sizes) if min(sizes) > 0 else 0
        print(f"   State {i}: {state} - requested points={centers.size}, actual used={n_pts}")

        if n_pts == 0:
            print("     ⚠️ Empty state - no data points")
            continue

        centers = centers[:n_pts]
        d2_vals = d2_vals[:n_pts]
        d2_err  = (d2_err[:n_pts] if has_err else np.full(n_pts, np.nan, dtype=float))

        # Ranges
        try:
            print(f"     Time range: {float(np.nanmin(centers))} to {float(np.nanmax(centers))}")
        except ValueError:
            print("     Time range: (all NaN)")

        try:
            print(f"     D2 range: {float(np.nanmin(d2_vals)):.4f} to {float(np.nanmax(d2_vals)):.4f}")
        except ValueError:
            print("     D2 range: (all NaN)")

        if np.any(np.isfinite(d2_err)):
            try:
                dmin, dmax = float(np.nanmin(d2_err)), float(np.nanmax(d2_err))
                print(f"     D2_ERR range: {dmin:.4f} to {dmax:.4f}")
            except ValueError:
                print("     D2_ERR range: (all NaN)")
        else:
            print("     D2_ERR: missing → filled with NaN")

        all_centers.append(centers)
        all_d2.append(d2_vals)
        all_d2err.append(d2_err)
        all_states.append(np.array([state]*n_pts, dtype=object))

    if len(all_centers) == 0:
        raise ValueError("No data found in any sleep state - all states are empty")

    # Concatenate
    all_centers = np.concatenate(all_centers)
    all_d2      = np.concatenate(all_d2)
    all_d2err   = np.concatenate(all_d2err)
    all_states  = np.concatenate(all_states)

    # DataFrame (include d2_err)
    df = pd.DataFrame({
        'center_s': all_centers.astype(float, copy=False),
        'd2':       all_d2.astype(float, copy=False),
        'd2_err':   all_d2err.astype(float, copy=False),
        'state':    all_states.astype(str)
    }).sort_values('center_s').reset_index(drop=True)

    print(f"\n✅ Complete timeline created:")
    print(f"📊 Total data points: {len(df)}")
    print(f"📊 Time range: {df['center_s'].min()} to {df['center_s'].max()}")
    print(f"📊 Duration: {(df['center_s'].max() - df['center_s'].min()):.1f} seconds")
    print(f"📊 State distribution:")
    print(df['state'].value_counts())

    # Empty-state tracker
    empty_states = []
    for ri in iter_results:
        st = _to_state_str(ri.get('state','UNKNOWN')) if hasattr(ri,'get') else \
             _to_state_str(ri['state']) if (isinstance(ri,dict) or hasattr(ri,'__getitem__')) and 'state' in ri else 'UNKNOWN'
        csz = np.size(np.atleast_1d(ri.get('center', []))) if hasattr(ri,'get') else \
              np.size(np.atleast_1d(ri['center'])) if (isinstance(ri,dict) or hasattr(ri,'__getitem__')) and 'center' in ri else 0
        if csz == 0:
            empty_states.append(st)

    meta = {
        "animal_name": animal_name,
        "day": day,
        "region": region,
        "path": mat_path,
        "params": params,
        "columns": list(df.columns),
        "shape": df.shape,
        "n_states": len(iter_results),
        "states_with_d2_err": n_states_with_err,
        "empty_states": empty_states
    }

    return df, meta



# Test the complete loading function
try:
    ca1_complete_df, ca1_complete_meta = load_d2_timeline_complete(basepath, "CA1")
    
    # pfc_complete_df, pfc_complete_meta = load_d2_timeline_complete(basepath, "PFC")

    
except Exception as e:
    print(f"❌ Failed: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*60)

Loading: day12_CA1_D2_time_resolved_error_bar.mat
📊 Found 3 result items (sleep states)
📊 Parameters: {'window_size': 30, 'step_size': 2, 'guard_sec': 5, 'deltaT': 0.04, 'order': 20, 'quant': 0.1, 'brain_region': 'CA1', 'n_cells': 92, 'total_spikes': 6291321}
   State 0: NREM - requested points=3937, actual used=3937
     Time range: 1713.0 to 27935.0
     D2 range: 0.0437 to 0.2399
     D2_ERR range: 0.0127 to 0.0306
   State 1: REM - requested points=0, actual used=0
     ⚠️ Empty state - no data points
   State 2: WAKE - requested points=7650, actual used=7650
     Time range: 21.0 to 28036.0
     D2 range: 0.0157 to 0.2676
     D2_ERR range: 0.0073 to 0.0365

✅ Complete timeline created:
📊 Total data points: 11587
📊 Time range: 21.0 to 28036.0
📊 Duration: 28015.0 seconds
📊 State distribution:
state
WAKE    7650
NREM    3937
Name: count, dtype: int64



In [66]:
ca1_complete_df

,center_s,d2,d2_err,state
0,21.0,0.108325,0.022439,WAKE
1,23.0,0.101031,0.022192,WAKE
2,25.0,0.091455,0.021008,WAKE
3,27.0,0.090963,0.020572,WAKE
4,29.0,0.085934,0.020024,WAKE
...,...,...,...,...
11582,28028.0,0.064112,0.017355,WAKE
11583,28030.0,0.079039,0.019131,WAKE
11584,28032.0,0.103376,0.020944,WAKE
11585,28034.0,0.103501,0.020867,WAKE


In [67]:
epoch_df

,name,startTime,stopTime,environment,behavioralParadigm,notes,manipulation,stimuli,basepath
0,presleep_230710_092540,0.0000,6372.72315,sleep,NaN,NaN,NaN,NaN,W:\data\PPP\PPP8\day12
1,T_long1_230710_111625,6372.7232,8515.37915,T_maze2,NaN,NaN,NaN,NaN,W:\data\PPP\PPP8\day12
2,midsleep_230710_115421,8515.3792,15122.09915,sleep,NaN,NaN,NaN,NaN,W:\data\PPP\PPP8\day12
3,T_long_maze1_230710_135143,15122.0992,17335.57115,T_maze1,1,NaN,NaN,NaN,W:\data\PPP\PPP8\day12
4,postsleep_230710_143427,17335.5712,28058.75835,sleep,NaN,09192023 postsleep tracking quality is not goo...,NaN,NaN,W:\data\PPP\PPP8\day12


In [68]:
import numpy as np
from scipy.signal import welch
import nelpy as nel

def _flatten_spike_times(spike_times):
    """Accepts list/tuple/array of arrays or a single 1D array, returns sorted 1D float array."""
    if isinstance(spike_times, (list, tuple)):
        # list of arrays or scalars
        if len(spike_times) == 0:
            return np.array([], dtype=float)
        if hasattr(spike_times[0], "__len__") and not np.isscalar(spike_times[0]):
            times = np.concatenate([np.asarray(st, dtype=float).ravel()
                                    for st in spike_times if st is not None and len(st) > 0])
        else:
            times = np.asarray(spike_times, dtype=float).ravel()
    else:
        arr = np.asarray(spike_times, dtype=object)
        if arr.dtype == object:
            times = np.concatenate([np.asarray(st, dtype=float).ravel()
                                    for st in arr if st is not None and len(st) > 0])
        else:
            times = arr.astype(float).ravel()

    times = times[np.isfinite(times)]
    if times.size == 0:
        raise ValueError("No valid spike times found after flattening.")
    return np.sort(times)

def bin_spikes(spike_times, bin_size=0.04, t_start=None, t_stop=None):
    st = _flatten_spike_times(spike_times)  # <-- key fix
    if t_start is None: t_start = float(np.floor(st.min()))
    if t_stop  is None: t_stop  = float(np.ceil(st.max()))
    edges = np.arange(t_start, t_stop + bin_size, bin_size)
    counts, _ = np.histogram(st, bins=edges)
    rates = counts / bin_size
    t = edges[:-1] + bin_size/2
    return t, rates

# def segments_from_epochs(t, x, epochs):
#     arr = epochs.data if isinstance(epochs, nel.EpochArray) else np.asarray(epochs)
#     return [x[(t >= s) & (t < e)] for s, e in arr if np.any((t >= s) & (t < e))]


In [69]:
# Create plot for random 200 second segment containing all states
import matplotlib.pyplot as plt
import numpy as np
import random

# First compute MUA from spike times
print("Computing MUA firing rate from spike times...")
print("="*50)

# Compute MUA firing rate with 0.04 second bins
time_bins, firing_rate = bin_spikes(spike_times, bin_size=0.04)

Computing MUA firing rate from spike times...


In [70]:
import numpy as np
import pandas as pd

# --- Inputs assumed to exist ---
# spike_times  # used earlier to make MUA; not needed again here
# time_bins, firing_rate  # from bin_spikes(..., bin_size=0.04)
# ca1_complete_df          # has a 'center_s' column

# 1) Make sure we have bin *centers* that match firing_rate length
tb = np.asarray(time_bins, float).ravel()
fr = np.asarray(firing_rate, float).ravel()

if tb.size == fr.size + 1:            # looks like edges → make centers
    t = 0.5 * (tb[:-1] + tb[1:])
elif tb.size == fr.size:               # already centers
    t = tb
else:
    raise ValueError("time_bins length must equal firing_rate length or be exactly one longer (edges).")

# 2) Clean + sort
ok = np.isfinite(t) & np.isfinite(fr)
t, fr = t[ok], fr[ok]
ordr = np.argsort(t)
t, fr = t[ordr], fr[ordr]

# 3) 30 s centered moving average of MUA (Hz)
dt = float(np.median(np.diff(t)))
win_s = 30.0
win_n = max(int(round(win_s / dt)), 1)

# Use pandas rolling for a centered window; allow some tolerance near edges
fr_series = pd.Series(fr, index=pd.Index(t, name="t"))
fr_30s = fr_series.rolling(window=win_n, center=True, min_periods=max(1, win_n//4)).mean().to_numpy()

# 4) Interpolate the smoothed MUA onto your D² centers
centers = np.asarray(ca1_complete_df["center_s"], float)
mua_at_centers = np.interp(centers, t, fr_30s, left=np.nan, right=np.nan)

# 5) Save to dataframe (Hz)
ca1_complete_df["MUA"] = mua_at_centers



SWR and barrage

In [71]:
import numpy as np
import scipy.io as sio      
import scipy
# Load ripple events
ripple_events = loading.load_ripples_events(basepath=basepath)
ripple_epochs = nel.EpochArray(np.array([ripple_events.start, ripple_events.stop]).T)
ripple_peaks=ripple_events.peaks
    # Load barrage events
ripple_epochs=nrem_epochs&ripple_epochs
try:
    # Load barrage events
    #brpath=r"W:\data\PPP\CA2\PPP7\day8"
    barrage_events = loading.load_barrage_events(basepath=basepath)  
    # Create barrage epochs
    barrage_epochs = nel.EpochArray(np.array([barrage_events.start, barrage_events.stop]).T)
    # Find barrages in NREM epochs
    barrages_in_nrem = barrage_epochs & nrem_epochs
    mat_path = os.path.join(basepath, f'{basename}.HSEn2.events.mat')

    data = scipy.io.loadmat(mat_path, struct_as_record=False, squeeze_me=True)
    barrage = data['HSEn2']
    barrage_timestamps= barrage.timestamps 
    barrage_epochs=nrem_epochs&barrage_epochs
except Exception as e:
    # General exception for unexpected errors
    print(f"No barrage file: {e}")
    barrage_epochs=[]
    # Extract start times


No barrage file: 'DataFrame' object has no attribute 'start'


D:\Documents\GitHub\neuro_py\neuro_py\io\loading.py:1036: UserWarning: No barrage file found for W:\data\PPP\PPP8\day12


d2 vs MUA

In [72]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import pickle
WIN_SEC = 30.0
HOP_SEC = 2.0
HALFWIN = WIN_SEC/2

# ---------- interval helpers ----------
def erode(iv, margin):
    iv = np.asarray(iv, float)
    if iv.size == 0: return iv.reshape(0,2)
    out = iv.copy(); out[:,0]+=margin; out[:,1]-=margin
    return out[out[:,1] > out[:,0]]

def dilate(iv, margin):
    iv = np.asarray(iv, float)
    if iv.size == 0: return iv.reshape(0,2)
    out = iv.copy(); out[:,0]-=margin; out[:,1]+=margin
    return out

def in_any_interval(t, iv):
    t = np.asarray(t, float)
    iv = np.asarray(iv, float)
    if t.size == 0 or iv.size == 0: return np.zeros(t.shape, bool)
    iv = iv[np.argsort(iv[:,0])]
    s, e = iv[:,0], iv[:,1]
    return (np.searchsorted(s, t, 'right') - np.searchsorted(e, t, 'right')) > 0

def mask_full_window_inside_state(df, state_name, halfwin=HALFWIN,
                                  time_col="center_s", state_col="state"):
    d = df.sort_values(time_col).copy()
    st = d[state_col].astype(str).str.upper().values
    t  = d[time_col].to_numpy(float)
    run_start = np.r_[0, np.where(st[1:] != st[:-1])[0] + 1]
    run_end   = np.r_[run_start[1:], len(d)]
    keep = np.zeros(len(d), bool)
    target = state_name.upper()
    for s, e in zip(run_start, run_end):
        if st[s] != target: 
            continue
        t0, t1 = t[s], t[e-1]
        keep[s:e] = (t[s:e] >= t0 + halfwin) & (t[s:e] <= t1 - halfwin)
    return keep

# ---------- MUA 30 s centered moving average on 0.04 s bins ----------
def moving_mean_centered(time_bins, firing_rate, win_sec=WIN_SEC):
    time_bins = np.asarray(time_bins, float)
    fr = np.asarray(firing_rate, float)
    dt = float(np.median(np.diff(time_bins)))
    L = int(round(win_sec / dt))           # kernel length in samples
    if L % 2 == 0: L += 1                  # use odd length for perfect centering
    k = np.ones(L, dtype=float)
    # handle NaNs/missing bins robustly
    x = fr.copy()
    w = np.isfinite(x).astype(float)
    x[~np.isfinite(x)] = 0.0
    num = np.convolve(x, k, mode="same")
    den = np.convolve(w, k, mode="same")
    mavg = np.where(den > 0, num/den, np.nan)
    valid = den >= L                        # full 30 s coverage only
    return mavg, valid, dt

# ---------- sample MUA mean at D² times ----------
def sample_at_times(source_t, source_y, target_t):
    source_t = np.asarray(source_t, float)
    target_t = np.asarray(target_t, float)
    idx = np.searchsorted(source_t, target_t)
    idx = np.clip(idx, 1, len(source_t)-1)
    left = idx-1
    right = idx
    # nearest neighbor to avoid smoothing
    nearer_left = np.abs(source_t[left] - target_t) <= np.abs(source_t[right] - target_t)
    nn_idx = np.where(nearer_left, left, right)
    return source_y[nn_idx]

# ---------- main: correlate per group ----------
def correlate_d2_mua_by_state_theta(
    ca_df,                       # has ['center_s','d2','state'] at 2 s steps
    time_bins, firing_rate,      # MUA at ~0.04 s bins
    theta_epochs,                # nel.EpochArray or (N,2) seconds
    *, prefer_rem_label="REM4",   # fall back to REM if REM4 not present
    make_scatter=False
):
    df = ca_df[["center_s","d2","state"]].copy()
    df["state_u"] = df["state"].astype(str).str.upper()
    rem_lbl = prefer_rem_label if (df["state_u"]==prefer_rem_label).any() else "REM"

    # 1) MUA 30 s moving average; sample at D² times
    mua30, mua_valid, dt = moving_mean_centered(time_bins, firing_rate, win_sec=WIN_SEC)
    mua_at_d2 = sample_at_times(time_bins, mua30, df["center_s"].to_numpy(float))
    valid_at_d2 = sample_at_times(time_bins, mua_valid.astype(float), df["center_s"].to_numpy(float)).astype(bool)

    # 2) full-window guards by state
    keep_wake = mask_full_window_inside_state(df, "WAKE")
    keep_nrem = mask_full_window_inside_state(df, "NREM")
    keep_rem  = mask_full_window_inside_state(df, rem_lbl)

    # 3) Wake θ vs non-θ (θ windows must fully contain the 30 s box; non-θ excludes any touch)
    theta_iv  = theta_epochs.data if hasattr(theta_epochs, "data") else np.asarray(theta_epochs, float)
    theta_ero = erode(theta_iv, HALFWIN)
    theta_dil = dilate(theta_iv, HALFWIN)

    wake_df = df[keep_wake & df["state_u"].eq("WAKE")].copy()
    t_wake  = wake_df["center_s"].to_numpy(float)
    in_theta_full = in_any_interval(t_wake, theta_ero)
    in_nontheta   = ~in_any_interval(t_wake, theta_dil)

    # assemble groups with indices in the original df
    idx_wake = df.index[keep_wake & df["state_u"].eq("WAKE")]
    idx_wake_theta    = idx_wake[in_theta_full]
    idx_wake_nontheta = idx_wake[in_nontheta]
    idx_nrem = df.index[keep_nrem & df["state_u"].eq("NREM")]
    idx_rem  = df.index[keep_rem  & df["state_u"].eq(rem_lbl)]

    groups = {
        "Wake θ": idx_wake_theta,
        "Wake non-θ": idx_wake_nontheta,
        "NREM": idx_nrem,
        rem_lbl: idx_rem,
    }

    # 4) compute correlations (Pearson & Spearman) within each group
    rows = []
    pairs_df_list = []
    for name, idx in groups.items():
        # also require MUA window validity
        idx = [i for i in idx if valid_at_d2[i]]
        if len(idx) < 3:
            rows.append({"group": name, "n": len(idx),
                        "pearson_r": np.nan, "pearson_p": np.nan,
                        "spearman_rho": np.nan, "spearman_p": np.nan})
            continue
        d2_vals  = df.loc[idx, "d2"].to_numpy(float)
        mua_vals = mua_at_d2[idx].astype(float)

        # drop NaNs
        ok = np.isfinite(d2_vals) & np.isfinite(mua_vals)
        d2_vals  = d2_vals[ok]
        mua_vals = mua_vals[ok]
        n = d2_vals.size

        if n < 3:
            rows.append({"group": name, "n": n,
                        "pearson_r": np.nan, "pearson_p": np.nan,
                        "spearman_rho": np.nan, "spearman_p": np.nan})
            continue

        r,  p  = stats.pearsonr(d2_vals, mua_vals)
        rho,sp = stats.spearmanr(d2_vals, mua_vals)
        rows.append({"group": name, "n": n,
                    "pearson_r": r, "pearson_p": p,
                    "spearman_rho": rho, "spearman_p": sp})

        pairs_df_list.append(pd.DataFrame({
            "time": df.loc[idx, "center_s"].to_numpy(float)[ok],
            "d2": d2_vals,
            "mua30": mua_vals,
            "group": name
        }))

        if make_scatter:
            plt.figure(figsize=(4.2,3.6))
            plt.scatter(d2_vals, mua_vals, s=6, alpha=0.35)
            ax = plt.gca()
            ax.grid(False)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
            plt.xlabel("D² (30 s, 2 s hop)")
            plt.ylabel("MUA mean (30 s centered)")
            plt.title(f"{name}: r={r:.2f} (p={p:.2g}) | ρ={rho:.2f} (p={sp:.2g})")
            plt.tight_layout(); plt.show()

    corr_results = pd.DataFrame(rows)
    pairs_df = pd.concat(pairs_df_list, ignore_index=True) if pairs_df_list else pd.DataFrame(columns=["time","d2","mua30","group"])
    return corr_results.sort_values("group"), pairs_df

# -------- run (CA1 example) --------
corr_df, pairs_df = correlate_d2_mua_by_state_theta(
    ca1_complete_df,         # has center_s, d2, state (2 s step)
    time_bins, firing_rate/n_cell,  # MUA at 0.04 s bins
    theta_epochs,            # nel.EpochArray with theta intervals
    prefer_rem_label="REM",
    make_scatter=False
)
def save_pairs_df(pairs_df, out_dir, animal, day, region="CA1"):
    pairs_df = pairs_df.copy()
    pairs_df["animal"] = str(animal)
    pairs_df["day"]    = str(day)
    pairs_df["region"] = str(region)
    pairs_df["animal_day"] = pairs_df["animal"] + "_" + pairs_df["day"]

    os.makedirs(out_dir, exist_ok=True)
    fname = f"{animal}_{day}_{region}_d2_mua_pairs.pkl"
    fpath = os.path.join(out_dir, fname)

    # Save as pickle
    pairs_df.to_pickle(fpath, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"✅ saved pairs (pickle): {fpath}  (n={len(pairs_df)})")

# Example:
save_pairs_df(pairs_df, r"D:\Documents\GitHub\Sleep_criticaltiy\data\cross_state\pairs", animal_name, day, region="CA1")
print(corr_df.to_string(index=False, float_format=lambda x: f"{x:.3g}"))


✅ saved pairs (pickle): D:\Documents\GitHub\Sleep_criticaltiy\data\cross_state\pairs\PPP8_day12_CA1_d2_mua_pairs.pkl  (n=6036)
     group    n  pearson_r  pearson_p  spearman_rho  spearman_p
      NREM 3633      0.332   1.98e-94         0.312    8.24e-83
       REM    0        NaN        NaN           NaN         NaN
Wake non-θ 1703     -0.283   8.93e-33        -0.214    4.37e-19
    Wake θ  700      0.123    0.00109        0.0985     0.00909
